# Dory family voice server (free, on Colab)

This runs OpenVoice v2 with MeloTTS on Colab's free GPU, so Dory can speak in a family member's voice.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Then run the cells in order (Runtime → Run all works too). Keep this tab open during the demo: if Colab
disconnects, just run all the cells again and redo step 4. Nothing breaks meanwhile: the app uses the
phone's own voice whenever this server is off.

OpenVoice v2 and MeloTTS are both MIT licensed: free for any use.

## 1. Install (about 3–5 minutes)

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"
# OpenVoice and MeloTTS pin very old library versions, so they're installed without them,
# followed by the libraries they actually need.
!rm -rf /content/OpenVoice /content/MeloTTS
!git clone -q --depth 1 https://github.com/myshell-ai/OpenVoice /content/OpenVoice
!git clone -q --depth 1 https://github.com/myshell-ai/MeloTTS /content/MeloTTS
!pip install -q --no-deps -e /content/OpenVoice -e /content/MeloTTS
!pip install -q librosa pydub eng_to_ipa inflect unidecode pypinyin cn2an jieba langid txtsplit cached_path "transformers<4.50" num2words unidic_lite mecab-python3 pykakasi fugashi g2p_en anyascii jamo "gruut[de,es,fr]" g2pkk loguru tqdm soundfile fastapi "uvicorn[standard]"
!python -c "import nltk; nltk.download("averaged_perceptron_tagger_eng", quiet=True); nltk.download("cmudict", quiet=True)"
# The OpenVoice v2 voice converter and base-voice files (MIT, from Hugging Face)
!mkdir -p /content/checkpoints_v2/converter /content/checkpoints_v2/base_speakers/ses
!for f in converter/config.json converter/checkpoint.pth base_speakers/ses/en-india.pth base_speakers/ses/en-us.pth base_speakers/ses/en-default.pth; do wget -q -O /content/checkpoints_v2/$f https://huggingface.co/myshell-ai/OpenVoiceV2/resolve/main/$f; done
!rm -rf /content/dory && git clone -q --depth 1 https://github.com/MeghaZachariah/Dory-Memory-Assistant /content/dory
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
print("Installed.")

## 2. Start the voice server (first time about 1–2 minutes: it downloads the English voice model)

In [ ]:
import os, secrets, subprocess, time, urllib.request, json

TOKEN = secrets.token_urlsafe(16)
env = {**os.environ, "VOICE_TOKEN": TOKEN, "VOICE_DIR": "/content/voices",
       "OPENVOICE_CKPT": "/content/checkpoints_v2", "VOICE_BASE_SPEAKER": "EN_INDIA"}
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "voice_server.server:app", "--host", "0.0.0.0", "--port", "8001"],
    cwd="/content/dory", env=env, stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)

for _ in range(300):
    time.sleep(2)
    try:
        if json.load(urllib.request.urlopen("http://localhost:8001/health"))["model_loaded"]:
            print("Voice server ready."); break
    except Exception:
        if server.poll() is not None:
            print(open("/content/server.log").read()[-3000:]); raise SystemExit("Server stopped, see the log above.")
else:
    print("Still loading... run this cell again in a minute.")

## 3. Give it a public address (free Cloudflare tunnel, no account needed)

In [ ]:
import re
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8001", "--no-autoupdate"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        URL = m.group(0); break
print("Voice server address:", URL or "not found yet, run this cell again")

## 4. Tell your Dory server where the voice is

Copy the command printed below and run it **on the laptop running the Dory server** (VS Code terminal).
Do this again every time you restart this notebook: the address changes each session.

In [ ]:
body = json.dumps({"url": URL, "token": TOKEN})
print("Git Bash / Mac / Linux:\n")
print(f"curl -X PUT http://localhost:8000/voice-server -H 'Content-Type: application/json' -d '{body}'\n")
print("PowerShell:\n")
print(f"Invoke-RestMethod -Method Put -Uri http://localhost:8000/voice-server -ContentType 'application/json' -Body '{body}'")

## 5. Done

In the app: **Settings → Family voice → Set up**, record the paragraph, then use **Test voice**.

The first time Dory says a new line it takes a second or two; lines it has said before are instant.